In [1]:
import os
import copy
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [4]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Setup complete.")
print("Device:", device)

✅ Setup complete.
Device: cuda


In [5]:
for class_name in os.listdir(TRAIN_DIR):
    class_folder = os.path.join(TRAIN_DIR, class_name)
    print(f"{class_name}: {len(os.listdir(class_folder))} images")

airplane: 70 images
bed: 70 images
bench: 70 images
bicycle: 70 images
bird: 70 images
bottle: 70 images
bowl: 70 images
bus: 70 images
cake: 70 images
car: 70 images
cat: 70 images
chair: 70 images
couch: 70 images
cow: 70 images
cup: 70 images
dog: 70 images
elephant: 70 images
horse: 70 images
motorcycle: 70 images
person: 70 images
pizza: 70 images
potted plant: 70 images
stop sign: 70 images
traffic light: 70 images
train: 70 images
truck: 70 images


In [6]:
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 15
NUM_CLASSES = 26

print("ENTERED INTO DATA AUGMENTATION")

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(20),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.85, 1.15)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("COMPLETED DATA AUGMENTATION")
print("=" * 32)

print("ENTER INTO DATA LOADERS")

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH,
    shuffle=False
)

print("COMPLETED DATA LOADERS")
print("=" * 32)

CLASS_NAMES = train_dataset.classes

print(f"\nNUMBER OF CLASSES: {len(CLASS_NAMES)}")
print(CLASS_NAMES)

ENTERED INTO DATA AUGMENTATION
COMPLETED DATA AUGMENTATION
ENTER INTO DATA LOADERS
COMPLETED DATA LOADERS

NUMBER OF CLASSES: 26
['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [7]:
weights = models.VGG16_Weights.DEFAULT
model = models.vgg16(weights=weights)

print("PRETRAINED VGG16 LOADED")

PRETRAINED VGG16 LOADED


In [8]:
for parameter in model.features.parameters():
    parameter.requires_grad = False

In [9]:
model.avgpool = nn.AdaptiveAvgPool2d((1, 1))

model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES)
)

In [10]:
model = model.to(device)

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=0.0001
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

print("\nMODEL COMPILED")


MODEL COMPILED


In [12]:
BEST_MODEL_PATH = "models/VGG16_best.pth"
PATIENCE = 5

In [13]:
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [14]:
print("\nSTARTING INITIAL TRAINING")

history = {
    "accuracy": [],
    "val_accuracy": [],
    "loss": [],
    "val_loss": []
}

best_val_accuracy = 0.0
epochs_without_improvement = 0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total
    val_loss /= val_total
    val_accuracy = val_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, BEST_MODEL_PATH)
        print("Best model saved.")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_model_state)


STARTING INITIAL TRAINING
Epoch [1/15] Loss: 3.2149 Accuracy: 0.0929 Val Loss: 3.1187 Val Accuracy: 0.1744
Best model saved.
Epoch [2/15] Loss: 3.0817 Accuracy: 0.2132 Val Loss: 2.9392 Val Accuracy: 0.2385
Best model saved.
Epoch [3/15] Loss: 2.8770 Accuracy: 0.2890 Val Loss: 2.6715 Val Accuracy: 0.2949
Best model saved.
Epoch [4/15] Loss: 2.6476 Accuracy: 0.3368 Val Loss: 2.4665 Val Accuracy: 0.3923
Best model saved.
Epoch [5/15] Loss: 2.3902 Accuracy: 0.3934 Val Loss: 2.2757 Val Accuracy: 0.4000
Best model saved.
Epoch [6/15] Loss: 2.1869 Accuracy: 0.4429 Val Loss: 2.1440 Val Accuracy: 0.3821
Epoch [7/15] Loss: 2.0395 Accuracy: 0.4555 Val Loss: 2.0686 Val Accuracy: 0.3923
Epoch [8/15] Loss: 1.9066 Accuracy: 0.4824 Val Loss: 1.9890 Val Accuracy: 0.4231
Best model saved.
Epoch [9/15] Loss: 1.7944 Accuracy: 0.5066 Val Loss: 1.9572 Val Accuracy: 0.4000
Epoch [10/15] Loss: 1.7164 Accuracy: 0.5264 Val Loss: 1.9319 Val Accuracy: 0.4103
Epoch [11/15] Loss: 1.6503 Accuracy: 0.5374 Val Loss: 

<All keys matched successfully>